# **Новые подходы к CATE: калибровка, конформные интервалы, выбор модели и CausalPFN**

Практика к модулю [«Гетерогенные эффекты»](https://ai-interpretability.school), урок «Как читать оценку и где ошибиться», шаг «Что используют сегодня: калибровка, конформные интервалы, выбор модели, foundation-модели».

В этом шаге четыре направления последних лет были строками таблицы. Здесь мы применим все четыре к одним данным с известным эффектом и проверим, что каждое обещает и где перестает работать:

1. причинная изотоническая калибровка ([van der Laan, Ulloa-Pérez, Carone и Luedtke, 2023](https://arxiv.org/abs/2302.14011)) чинит числа $\hat\tau(x)$, не меняя порядка объектов;
2. взвешенные конформные интервалы ([Lei и Candès, 2021](https://arxiv.org/abs/2006.06138)) дают интервал для ненаблюдаемого исхода и для эффекта отдельного объекта с покрытием в среднем по объектам;
3. CausalPFN ([Balazadeh et al., 2025](https://arxiv.org/abs/2506.07918)) оценивает эффект заранее обученным трансформером, без обучения на наших данных;
4. R-score ([Nie и Wager, 2021](https://arxiv.org/abs/1712.04912); реализация `econml.score.RScorer`) выбирает оценку CATE без истинного эффекта — один из классов метрик из обзоров [Curth и van der Schaar (2023)](https://arxiv.org/abs/2302.02923) и [Mahajan et al. (2024)](https://arxiv.org/abs/2211.01939).

План тетради:

1. данные: наблюдательное исследование с конфаундингом и известным $\tau(x)$;
2. четыре оценки CATE: S-learner на бустинге и на лассо, T-learner, причинный лес;
3. причинная изотоническая калибровка;
4. взвешенные конформные интервалы для $Y(1)$, $Y(0)$ и для эффекта объекта;
5. CausalPFN;
6. выбор модели по R-score;
7. где ломается: R-score на малой выборке и CausalPFN со скрытым конфаундером;
8. реализации.

Тетрадь идет на процессоре; весь прогон занимает одну-две минуты. Веса CausalPFN (75 МБ) скачиваются с Hugging Face при первом вызове.

In [ ]:
!pip install -q econml==0.17.0 causalpfn==0.1.5

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold
from econml.dml import CausalForestDML
from econml.score import RScorer
from causalpfn import CATEEstimator

warnings.filterwarnings('ignore', message='.*encountered in matmul')   # ложные предупреждения numpy на некоторых процессорах
SEED = 0
torch.set_num_threads(1)     # один поток: одинаковый порядок сложений в CausalPFN при каждом запуске
pd.set_option('display.width', 140)

## 1. Данные

Наблюдательное исследование: 7000 объектов, пять признаков $X\sim\mathcal N(0, I)$. Воздействие получают не случайно: склонность $e(x)=\sigma(0.6\,x_1-0.3\,x_2)$ лежит от 0.09 до 0.93, а $x_1$ входит и в исход без воздействия $\mu_0(x)=2x_1+x_2^2$ — это конфаундер. Эффект $\tau(x)=1+1.5\,x_0+0.5\max(x_3, 0)$. Шум у исхода зависит от $x_0$: стандартное отклонение $0.5+0.5|x_0|$, чтобы интервалы для объектов с большим $|x_0|$ были шире. Оба потенциальных исхода генерируются с независимым шумом, поэтому эффект объекта $Y_i(1)-Y_i(0)$ — не то же, что $\tau(x_i)$: в нем есть и шум двух исходов.

Три части: на первых 3000 объектах (`tr`) учатся оценки CATE и модели квантилей, на следующих 2000 (`ca`) — калибровка, конформная поправка и выбор модели, на последних 2000 (`te`) все проверяется по истинным $\tau(x)$, $Y(1)$ и $Y(0)$. На настоящих данных видны только $X$, $T$ и $Y$.

In [ ]:
def simulate(n, r):
    X = r.normal(size=(n, 5))
    e = 1 / (1 + np.exp(-(0.6 * X[:, 1] - 0.3 * X[:, 2])))
    T = r.binomial(1, e)
    tau = 1 + 1.5 * X[:, 0] + 0.5 * np.maximum(X[:, 3], 0)
    sd = 0.5 + 0.5 * np.abs(X[:, 0])
    mu0 = 2 * X[:, 1] + X[:, 2] ** 2
    Y0 = mu0 + sd * r.normal(size=n)
    Y1 = mu0 + tau + sd * r.normal(size=n)
    return X, T, np.where(T == 1, Y1, Y0), tau, e, Y0, Y1


X, T, Y, tau, e, Y0, Y1 = simulate(7000, np.random.default_rng(SEED))
tr, ca, te = np.arange(3000), np.arange(3000, 5000), np.arange(5000, 7000)
naive = Y[te][T[te] == 1].mean() - Y[te][T[te] == 0].mean()
print('доля воздействованных среди 7000 объектов:', round(T.mean(), 3), '| склонность от', round(e.min(), 3), 'до', round(e.max(), 3))
print('истинный средний эффект на te:', round(tau[te].mean(), 3), '| наивная разность средних:', round(naive, 3))

## 2. Четыре оценки CATE

Берем четыре оценки, которые уже встречались в модуле, и одну из них нарочно портим:

- S-learner на градиентном бустинге: одна модель исхода, воздействие — признак;
- S-learner на лассо с взаимодействиями $x\cdot t$ и сильной регуляризацией (`alpha=0.3`): штраф стягивает коэффициенты при $t$ и $x\cdot t$ к нулю, и эффект выходит меньше настоящего — та самая поломка S-learner из урока «Мета-обучатели: S, T и X»;
- T-learner на градиентном бустинге: по модели на группу;
- причинный лес `CausalForestDML`.

Ошибку меряем, как в остальных тетрадях модуля: $\mathrm{PEHE}=\sqrt{\tfrac1n\sum_i(\hat\tau(x_i)-\tau(x_i))^2}$ на `te`. Рядом — ошибка калибровки: объекты делим на 20 равных групп по прогнозу $\hat\tau$ и складываем квадраты разностей между средним истинным и средним предсказанным эффектом в группе с весом доли группы. Это оценка величины $\mathrm{CAL}(\hat\tau)=\mathbb E\bigl[(\mathbb E[\tau(X)\mid\hat\tau(X)]-\hat\tau(X))^2\bigr]$ из статьи van der Laan et al.; посчитать ее можно только при известном $\tau$.

In [ ]:
def gb():
    return GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=SEED)


def gbc():
    return GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=SEED)


def pehe(p, t):
    return np.sqrt(np.mean((p - t) ** 2))


def cal_error(p, t, k=20):
    groups = np.array_split(np.argsort(p, kind='stable'), k)
    return sum(len(g) / len(p) * (t[g].mean() - p[g].mean()) ** 2 for g in groups)


def inter(Z, t):
    return np.c_[Z, t, Z * t[:, None]]


s_gb = gb().fit(np.c_[X[tr], T[tr]], Y[tr])
s_lasso = Lasso(alpha=0.3).fit(inter(X[tr], T[tr]), Y[tr])
m1 = gb().fit(X[tr][T[tr] == 1], Y[tr][T[tr] == 1])
m0 = gb().fit(X[tr][T[tr] == 0], Y[tr][T[tr] == 0])
cf = CausalForestDML(model_y=gb(), model_t=gbc(), discrete_treatment=True, n_estimators=500,
                     min_samples_leaf=40, random_state=SEED).fit(Y[tr], T[tr], X=X[tr])
one, zero = lambda Z: np.ones(len(Z)), lambda Z: np.zeros(len(Z))
models = {
    'S-learner, GB': lambda Z: s_gb.predict(np.c_[Z, one(Z)]) - s_gb.predict(np.c_[Z, zero(Z)]),
    'S-learner, lasso': lambda Z: s_lasso.predict(inter(Z, one(Z))) - s_lasso.predict(inter(Z, zero(Z))),
    'T-learner, GB': lambda Z: m1.predict(Z) - m0.predict(Z),
    'causal forest': lambda Z: cf.effect(Z),
}
pred = {k: {'ca': f(X[ca]), 'te': f(X[te])} for k, f in models.items()}
print(pd.DataFrame({k: [pehe(p['te'], tau[te]), cal_error(p['te'], tau[te]), p['te'].mean() - tau[te].mean()] for k, p in pred.items()},
                   index=['PEHE', 'CAL', 'ошибка среднего']).T.round(3))

## 3. Причинная изотоническая калибровка

Прогноз $\hat\tau$ откалиброван, если среди объектов с прогнозом $v$ средний настоящий эффект равен $v$. Истинного эффекта нет, но есть его несмещенный заменитель — дважды робастный псевдоисход DR-learner:
$$\hat\varphi_i=\hat\mu_1(x_i)-\hat\mu_0(x_i)+\frac{t_i-\hat e(x_i)}{\hat e(x_i)(1-\hat e(x_i))}\bigl(y_i-\hat\mu_{t_i}(x_i)\bigr).$$
Алгоритм 1 статьи van der Laan et al.: посчитать $\hat\varphi_i$ на выборке для калибровки, с моделями исхода и склонности из других фолдов этой выборки (вариант с cross-fitting, алгоритм 4 статьи), и найти неубывающую функцию $\theta$, ближе всего к псевдоисходам в смысле квадрата ошибки:
$$\hat\theta=\arg\min_{\theta\ \text{не убывает}}\ \sum_{i\in\mathrm{ca}}\bigl(\hat\varphi_i-\theta(\hat\tau(x_i))\bigr)^2 .$$
Откалиброванный прогноз — $\hat\theta(\hat\tau(x))$. Это обычная изотоническая регрессия `sklearn.isotonic.IsotonicRegression` псевдоисхода на прогноз. Функция неубывающая, поэтому порядок объектов сохраняется, но соседние прогнозы могут получить одно и то же значение.

In [ ]:
def pseudo_outcome(Xc, Tc, Yc, folds=5):
    phi = np.zeros(len(Yc))
    for a, b in KFold(folds, shuffle=True, random_state=SEED).split(Xc):
        g1 = gb().fit(Xc[a][Tc[a] == 1], Yc[a][Tc[a] == 1]).predict(Xc[b])
        g0 = gb().fit(Xc[a][Tc[a] == 0], Yc[a][Tc[a] == 0]).predict(Xc[b])
        p = np.clip(gbc().fit(Xc[a], Tc[a]).predict_proba(Xc[b])[:, 1], 0.01, 0.99)
        phi[b] = g1 - g0 + (Tc[b] - p) / (p * (1 - p)) * (Yc[b] - np.where(Tc[b] == 1, g1, g0))
    return phi


phi = pseudo_outcome(X[ca], T[ca], Y[ca])
print('среднее псевдоисхода на ca:', round(phi.mean(), 3), '| истинный средний эффект на ca:', round(tau[ca].mean(), 3))

**Задание 1. Напишите функцию `calibrate(p_ca, p_te)`: изотоническая регрессия псевдоисхода `phi` на прогноз `p_ca` на выборке `ca` (`IsotonicRegression(out_of_bounds='clip')`), результат — откалиброванный прогноз для `p_te`. Откалибруйте S-learner на лассо. Чему равны PEHE и CAL до и после калибровки и средний эффект до и после? Нарисуйте прогноз против истинного $\tau$ до и после.**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

**Задание 2. Откалибруйте той же функцией остальные три оценки. Выведите таблицу: PEHE и CAL до и после калибровки и число разных значений откалиброванного прогноза. У каких оценок CAL падает, а у каких растет PEHE? Как это согласуется с разложением $\mathrm{MSE}=\mathrm{CAL}+\mathrm{DIS}$ из статьи?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 4. Взвешенные конформные интервалы

Исход $Y(1)$ известен только у воздействованных, а интервал для него нужен у всех. Это задача со сдвигом распределения признаков: модель учится на $P_{X\mid T=1}$, а применяется на $P_X$. Lei и Candès строят интервал взвешенным split-CQR (алгоритм 1 статьи):

1. на обучающей части группы $T=1$ учим две модели квантилей исхода, $\hat q_{\alpha/2}(x)$ и $\hat q_{1-\alpha/2}(x)$;
2. на калибровочной части группы $T=1$ считаем оценку несоответствия $V_i=\max\{\hat q_{\alpha/2}(x_i)-y_i,\ y_i-\hat q_{1-\alpha/2}(x_i)\}$;
3. каждому калибровочному объекту даем вес $w_1(x_i)=1/\hat e(x_i)$ (таблица 1 статьи: цель — вся совокупность), новому объекту $x$ — вес $w_1(x)$;
4. $\eta(x)$ — квантиль уровня $1-\alpha$ распределения, которое ставит на $V_i$ массу $w_1(x_i)/(\sum_j w_1(x_j)+w_1(x))$, а на $+\infty$ — остаток $w_1(x)/(\sum_j w_1(x_j)+w_1(x))$;
5. интервал $[\hat q_{\alpha/2}(x)-\eta(x),\ \hat q_{1-\alpha/2}(x)+\eta(x)]$.

Для $Y(0)$ то же на контрольной группе с весом $w_0(x)=1/(1-\hat e(x))$. Без весов (все $w=1$) это обычный split-CQR. При верных весах покрытие не ниже $1-\alpha$ в среднем по объектам, но не для каждого $x$.

Интервал для эффекта объекта $Y(1)-Y(0)$ в разделе 4.1 статьи строят «наивно»: интервалы для $Y(1)$ и $Y(0)$ уровня $1-\alpha/2$ каждый и разность их концов, $[\hat Y^L(1)-\hat Y^R(0),\ \hat Y^R(1)-\hat Y^L(0)]$. Покрытие не ниже $1-\alpha$ гарантировано, но с запасом; вложенная схема из раздела 4.2 дает интервалы уже.

In [ ]:
propensity = gbc().fit(X[tr], T[tr])


def e_hat(Z):
    return propensity.predict_proba(Z)[:, 1]


def quantile_gb(q):
    return GradientBoostingRegressor(loss='quantile', alpha=q, n_estimators=200, max_depth=3, random_state=SEED)

**Задание 3. Напишите функцию `conformal(t, alpha, weighted)`: она учит модели квантилей `quantile_gb` на объектах `tr` группы `t`, считает оценки несоответствия на объектах `ca` той же группы и возвращает функцию, которая по матрице признаков выдает нижние и верхние концы интервалов (с весами $1/\hat e$ или $1/(1-\hat e)$ при `weighted=True`, без весов при `weighted=False`). Постройте 90-процентные интервалы для $Y(1)$ и $Y(0)$ на `te` без весов и с весами и выведите долю объектов, у которых интервал накрыл настоящий потенциальный исход, — среди всех и среди тех, у кого этот исход не наблюдался. Затем постройте наивный 90-процентный интервал для $Y(1)-Y(0)$ с весами: какая у него доля накрытых объектов и средняя ширина интервала?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

**Задание 4. Покрытие в среднем — не покрытие у каждого объекта. Разбейте `te` на пять равных групп по $x_0$ (`np.argsort(X[te, 0])`, `np.array_split(..., 5)`): первая группа — 400 объектов с наименьшим $x_0$. Для 90-процентных интервалов для $Y(1)$ с весами выведите в каждой группе долю накрытых объектов и среднюю ширину интервала. Сколько из 400 объектов первой группы накрыто и почему покрытие ниже 90% именно там?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 5. CausalPFN

CausalPFN — трансформер, который авторы один раз обучили на большой библиотеке симулированных процессов с известным эффектом, где выполнено допущение об отсутствии скрытых конфаундеров. На новых данных он не учится: таблица `tr` с $X$, $T$, $Y$ подается ему как контекст, и он выдает оценку эффекта для новых объектов, как prior-fitted network выдает прогноз. `CATEEstimator.fit` только запоминает контекст и загружает веса; работа идет в `estimate_cate`.

Модель и код — под лицензией Apache-2.0. Предупреждение Hugging Face о запросах без токена можно не замечать: веса открытые.

**Задание 5. Оцените CATE с CausalPFN (`CATEEstimator(device='cpu')`, контекст — `tr`) на `ca` и на `te`. Чему равны PEHE, CAL и ошибка среднего эффекта на `te`, сколько секунд заняла оценка? Сравните с четырьмя оценками из раздела 2.**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 6. Выбор модели по R-score

На реальных данных PEHE не посчитать. Метрика-заменитель R-score построена на функции потерь R-learner: на выборке для выбора `ca` с cross-fitting оцениваются $\hat m(x)=\mathbb E[Y\mid X=x]$ и $\hat e(x)$, и оценка $\hat\tau$ получает потерю $\frac1n\sum_i\bigl((y_i-\hat m(x_i))-\hat\tau(x_i)(t_i-\hat e(x_i))\bigr)^2$. `econml.score.RScorer` сравнивает ее с потерей лучшего постоянного эффекта: $\text{R-score}=1-\text{потеря}(\hat\tau)/\text{потеря}(\text{const})$; больше — лучше, отрицательное значение — хуже постоянного эффекта. Curth и van der Schaar относят такие метрики к классу «по псевдоисходу» и показывают, что любая метрика-заменитель тянет к оценкам, похожим на свои вспомогательные модели.

`RScorer.score` ждет объект с методом `const_marginal_effect`, поэтому готовые прогнозы оборачиваем в маленький класс.

In [ ]:
class Fixed:
    # готовые прогнозы на строках ca в виде, который понимает RScorer
    def __init__(self, values):
        self.values = values

    def const_marginal_effect(self, Z):
        return self.values

**Задание 6. Обучите `RScorer(model_y=gb(), model_t=gbc(), discrete_treatment=True, cv=5, random_state=SEED)` на `ca` и посчитайте R-score пяти оценок (четыре из раздела 2 и CausalPFN). Выведите таблицу R-score и PEHE на `te` и ранговую корреляцию Спирмена между R-score и $-\mathrm{PEHE}$. Какую оценку выбирает R-score и совпадает ли его порядок с порядком по PEHE?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 7. Где ломается

### 7.1. R-score на малой выборке

R-score — тоже оценка по выборке, со своим шумом: и вспомогательные модели $\hat m$, $\hat e$, и сама потеря зависят от того, какие объекты попали в выборку для выбора. В разделе 6 у R-score были 2000 объектов. Посмотрим, что он делает на пяти непересекающихся частях по 400.

**Задание 7. Разбейте `ca` на пять частей по 400 (`np.array_split(np.random.default_rng(SEED).permutation(len(ca)), 5)` — позиции внутри `ca`), на каждой обучите такой же `RScorer` и посчитайте R-score пяти оценок. Какую оценку R-score выбирает на каждой части? На скольких частях выбор совпал с выбором на всей `ca`, и насколько на них близки R-score первой и второй оценок?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

### 7.2. CausalPFN и скрытый конфаундер

Гарантии CausalPFN выведены при допущении, что все конфаундеры среди признаков: на таких процессах модель обучали. Проверим, что будет, если конфаундер $x_1$ спрятать — убрать из признаков, как будто его не записали. Влияние $x_1$ на воздействие и на исход при этом остается.

**Задание 8. Уберите $x_1$ из признаков (`keep = [0, 2, 3, 4]`) и оцените CATE на `te` с CausalPFN (контекст — `tr` без $x_1$) и причинным лесом из раздела 2, обученным без $x_1$. Чему равны PEHE и ошибка среднего эффекта у обоих? Сравните с наивной разностью средних из раздела 1.**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

## 8. Реализации

- Причинная изотоническая калибровка: R-пакет авторов [causalCalibration](https://github.com/Larsvanderlaan/causalCalibration) (лицензия MIT). В Python — как здесь: псевдоисход DR-learner и `sklearn.isotonic.IsotonicRegression`; в статье есть и cross-calibration, где калибруется прогноз, полученный cross-fitting на всех данных, без отдельной выборки для калибровки.
- Конформные интервалы для потенциальных исходов и ITE: R-пакет авторов [cfcausal](https://github.com/lihualei71/cfcausal) (лицензия MIT) — взвешенный CQR, наивная и вложенная схемы. Готовой реализации на Python для эффекта нет; взвешенный split-CQR — функция на 15 строк, как выше.
- Выбор модели: [`econml.score.RScorer`](https://github.com/py-why/EconML) (лицензия MIT) — R-score, а также `RScorer.best_model` и `RScorer.ensemble` для выбора и смеси оценок.
- CausalPFN: [`causalpfn`](https://github.com/vdblm/CausalPFN) (лицензия Apache-2.0): `CATEEstimator` и `ATEEstimator`; в `CATEEstimator` есть еще калибровка неопределенности (`calibrate=True`) и интервалы `estimate_cate_CI`.

## Что унести из тетради

- **Калибровка чинит числа, а не порядок.** Испорченный регуляризацией S-learner после изотонической калибровки по псевдоисходу из ошибочного по уровню стал одним из лучших; у оценок, которые уже были почти откалиброваны, CAL падает, а PEHE немного растет: у откалиброванного прогноза всего 68–83 разных значения.
- **Конформный интервал покрывает в среднем.** Взвешивание по склонности переносит покрытие с наблюдаемой группы на всю совокупность, но в области с большим шумом доля накрытых ниже заявленной; наивный интервал для эффекта объекта надежен и широк.
- **CausalPFN без обучения на наших данных дал наименьший PEHE** — при выполненном допущении об отсутствии скрытых конфаундеров. Без конфаундера среди признаков он ошибается так же, как причинный лес: допущение не проверяется ни одной из оценок.
- **R-score на 2000 объектах упорядочил оценки так же, как PEHE,** но на 400 объектах выбор неустойчив там, где R-score двух оценок CATE различаются в третьем или четвертом знаке.

Источники:
1. [van der Laan, Ulloa-Pérez, Carone и Luedtke, 2023. Causal isotonic calibration for heterogeneous treatment effects](https://arxiv.org/abs/2302.14011)
2. [Lei и Candès, 2021. Conformal inference of counterfactuals and individual treatment effects](https://arxiv.org/abs/2006.06138)
3. [Nie и Wager, 2021. Quasi-oracle estimation of heterogeneous treatment effects](https://arxiv.org/abs/1712.04912)
4. [Curth и van der Schaar, 2023. In search of insights, not magic bullets: towards demystification of the model selection dilemma in heterogeneous treatment effect estimation](https://arxiv.org/abs/2302.02923)
5. [Mahajan et al., 2024. Empirical analysis of model selection for heterogeneous causal effect estimation](https://arxiv.org/abs/2211.01939)
6. [Balazadeh et al., 2025. CausalPFN: Amortized causal effect estimation via in-context learning](https://arxiv.org/abs/2506.07918)